# Extracting of lakes
This function:

- Loop on all polygons in the shape to find the grid's cells they intersect with

- Extract a list of IDs of polygons for each cell

- Loop on the grids, find the correct shape and extract the data

## Lake extraction

loop on the polygons

In [ ]:
# importing all libraries needed for the extraction
import zipfile
import geopandas as gpd
import os
import re
import pandas as pd
from tqdm import tqdm

In [ ]:
# connection to the drive where the data is kept and where the results will be saved
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Here the parameters for the code to run smoothly are defined
# this code will loop on all the grids saved in a zipped folder (less place used and still efficient)

# === PARAMETERS ===
# path to the lakes folder
POLY_FOLDER = "/content/Lakes_ar"
# ID field identifiying the lakes
ID_FIELD = "Hylak_id"
# path to the folder where you want the results to be saved as .csv in your drive
OUTPUT_FOLDER = "/content/drive/MyDrive/polygones_extraction"
# Path to the zip file containing the grids of the different zones
ZIP_PATH = "/content/drive/MyDrive/Zone/grid_001_zone3.zip"

# if the output folder does not exist, it is created
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [ ]:
# This function loop on all the lakes polygons and register all the grids one polygon intersects. At the end, the results are articulated based on lat and long.
def indexation_polygones(grille_zone_path, shapefile_path, champ_id):
    # Load data
    grille = gpd.read_file(grille_zone_path)
    polygones = gpd.read_file(shapefile_path)

    # Get the grid's spatial index
    sindex = grille.sindex

    # Create dictionary to get all results
    cell_dict = {}

    # Following progress
    tqdm.pandas(desc="Polygons' processing")

    # loop on the lake's polygons
    for poly in tqdm(polygones.itertuples(index=False), total=len(polygones), desc="Polygons"):
        # get the polygon's geometry
        geom = poly.geometry
        # get the id
        val = getattr(poly, champ_id)

        # get the intersection between the polygon and the cells
        possible_matches_index = list(sindex.intersection(geom.bounds))
        intersecting = grille.iloc[possible_matches_index]
        intersecting = intersecting[intersecting.intersects(geom)]

        # save the intersections fo each cell
        for idx in intersecting.index:
            if idx not in cell_dict:
                cell_dict[idx] = []
            cell_dict[idx].append(val)

    # Results saved as dataframe
    grille["lake_ids"] = grille.index.map(lambda idx: cell_dict.get(idx, []))
    grille["latitude"] = round(grille.centroid.y,3)
    grille["longitude"] = round(grille.centroid.x, 3)

    df = grille[["latitude", "longitude", "lake_ids"]].copy()
    # This takes care of the possible duplicates of lat and long, and concatenate the values
    df = df.groupby(["latitude", "longitude"], as_index=False).agg({
        "lake_ids": lambda lists: sorted(set(sum(lists, [])))
    })

    # Convert the list of ids to string (ex: 14777|1369851)
    df["lake_ids"] = df["lake_ids"].apply(lambda x: "|".join(map(str, x)))
    # return the results dataframe
    return df


In [ ]:
# Search the zip file with the grids
with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
    # Only choose the files in format .gpkg
    gpkg_files = [f for f in archive.namelist() if f.endswith(".gpkg")]

    # Loop on all the grids
    for filename in gpkg_files:

        # Extract the zone's number from the name of the file
        match = re.search(r"grid_001_zone(\d+)", filename) # rename the search depending on the filename format
        if match:
            zone_id = int(match.group(1))
        else:
            print(f"❌ Impossible to find the zone's number in this file : {filename}")
            continue

        print(f"\n➡️ Extraction zone_id {zone_id}")

        # Read the 0.01 grid and the lakes shape of the same zone
        GRILLE_PATH = f"/vsizip/{ZIP_PATH}/{filename}"
        POLY_PATH = os.path.join(POLY_FOLDER, f"lakes_zone_{zone_id}.shp")

        # Get the list of all the ids
        try:
            df_zone = indexation_polygones(GRILLE_PATH, POLY_PATH, ID_FIELD)
        except Exception as e:
            print(f"❌ Error processing for {zone_id} : {e}")
            continue

        # Save as .csv
        try:
            # create a double entry matrix based on lat and long
            pivoted = df_zone.pivot(index="latitude", columns="longitude", values="lake_ids")
            # keep the lat and long in the right order (N-S and W-E)
            pivoted = pivoted.sort_index(ascending=False)
            # save as csv
            pivoted_csv = os.path.join(OUTPUT_FOLDER, f"Extraction_{ID_FIELD}_zone_{zone_id}_pivoted.csv")
            pivoted.to_csv(pivoted_csv)
            print(f"✅ Double entry table saved at :  {pivoted_csv}")
        except Exception as e:
            # in case there was a problem to create the double entry table, the dataframe will be saved as is
            output_csv = os.path.join(OUTPUT_FOLDER, f"Extraction_{ID_FIELD}_zone_{zone_id}.csv")
            df_zone.to_csv(output_csv, index=False)
            print(f"❌ Double entry was not created : {e}\n✅ Data still saved in brut at {output_csv}.")
